In [ ]:
import pandas as pd
import numpy as np
from surprise.model_selection import train_test_split, GridSearchCV
from surprise import accuracy
from typing import Any, List
from sklearn.metrics.pairwise import cosine_similarity
import warnings
from surprise import SVD, Dataset, Reader, accuracy, Prediction
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Input, Dense, Flatten, Concatenate
from tensorflow.keras.models import Model

# Suppress all warnings
warnings.filterwarnings("ignore")

In [3]:
ratings = pd.read_csv('ratings.csv')
ratings.head(3)

,Unnamed: 0,user,user_id,Title,movie_id,Rating,Genre List,Director List,Writer List,Actors List,Plot,Language List,Country List
0,0,007filmreviwer,0,Den of Thieves 2: Pantera,11919.0,4.0,"['Action', 'Crime', 'Drama']",['Christian Gudegast'],['Christian Gudegast'],"['Gerard Butler', ""O'Shea Jackson Jr."", 'Evin ...",Big Nick is back on the hunt in Europe and clo...,['English'],"['United States', 'Canada', 'Spain']"
1,1,007filmreviwer,0,Sonic the Hedgehog 3,11941.0,3.5,"['Action', 'Adventure', 'Comedy']",['Jeff Fowler'],"['Pat Casey', 'Josh Miller', 'John Whittington']","['Jim Carrey', 'Ben Schwartz', 'Keanu Reeves']","Sonic, Knuckles, and Tails reunite against a p...",['English'],"['United States', 'Japan']"
2,2,007filmreviwer,0,Kraven the Hunter,11943.0,1.5,"['Action', 'Thriller']",['J.C. Chandor'],"['Richard Wenk', 'Art Marcum', 'Matt Holloway']","['Aaron Taylor-Johnson', 'Ariana DeBose', 'Fre...",Kraven's complex relationship with his ruthles...,"['English', 'Russian', 'Turkish']","['United States', 'Iceland', 'Canada', 'United..."


In [ ]:
# Load the dataset using the Surprise reader
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['user_id', 'movie_id', 'Rating']], reader)

# Define the parameter grid
param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs': [20, 30],
    'lr_all': [0.002, 0.005, 0.01],
    'reg_all': [0.02, 0.1]
}

# Perform grid search using RMSE as the evaluation metric
grid_search = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3)
grid_search.fit(data)

# Use the best parameters from grid search to train the SVD model
best_params = grid_search.best_params['rmse']
print("Best parameters:", best_params)

# Split data into train and validation sets manually
# Step 1: Split into training (70%) and temp (30%)
train_set, temp_set = train_test_split(data, test_size=0.3)

# Step 2: Split temp_set into validation (15%) and test (15%)
val_set, test_set = train_test_split(temp_set, test_size=0.5)

# Train the SVD model with the best parameters
svd_model = SVD(n_factors=best_params['n_factors'],
                n_epochs=best_params['n_epochs'],
                lr_all=best_params['lr_all'],
                reg_all=best_params['reg_all'])

print("Training the fine-tuned SVD model...")
svd_model.fit(train_set)

# Evaluate on the validation set
predictions = svd_model.test(val_set)
print("Final RMSE with tuned SVD model:")
accuracy.rmse(predictions)

Best parameters: {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}
Training the fine-tuned SVD model...
Final RMSE with tuned SVD model:
RMSE: 0.7813


0.7812951050540867

In [74]:
# Extract raw training data from the Surprise train_set
trainset_data = train_set.build_testset()
# Convert to a DataFrame
train_df = pd.DataFrame(trainset_data, columns=['user_id', 'item_id', 'rating']).dropna()

val_df = pd.DataFrame(val_set).dropna()
val_df.columns = ['user_id', 'item_id', 'rating']

test_df = pd.DataFrame(test_set).dropna()
test_df.columns = ['user_id', 'item_id', 'rating']

In [ ]:
ratings_df = ratings.dropna()

# Map user and item IDs to contiguous indices using pd.factorize()
train_df['user_index'] = pd.factorize(train_df['user_id'])[0]
train_df['movie_index'] = pd.factorize(train_df['item_id'])[0]
val_df['user_index'] = pd.factorize(val_df['user_id'])[0]
val_df['movie_index'] = pd.factorize(val_df['item_id'])[0]
test_df['user_index'] = pd.factorize(test_df['user_id'])[0]
test_df['movie_index'] = pd.factorize(test_df['item_id'])[0]

num_users = train_df['user_index'].nunique()
num_items = train_df['movie_index'].nunique()

In [ ]:
# Hyperparameters
embedding_size = 50

# User input and embedding
user_input = Input(shape=(1,), name="user_input")
user_embedding = Embedding(input_dim=num_users, output_dim=embedding_size, name="user_embedding")(user_input)
user_vec = Flatten()(user_embedding)

# Movie input and embedding
movie_input = Input(shape=(1,), name="movie_input")
movie_embedding = Embedding(input_dim=num_items, output_dim=embedding_size, name="movie_embedding")(movie_input)
movie_vec = Flatten()(movie_embedding)

# Concatenate user and movie embeddings
concatenated = Concatenate()([user_vec, movie_vec])

# Fully connected layers
dense1 = Dense(128, activation='relu')(concatenated)
dense2 = Dense(64, activation='relu')(dense1)
dense3 = Dense(32, activation='relu')(dense2)

# Output layer (rating prediction)
output = Dense(1, activation='linear')(dense3)

# Define and compile the model
model = Model(inputs=[user_input, movie_input], outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Display model summary
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 50)     │     49,900 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_embedding     │ (None, 1, 50)     │  1,164,650 │ movie_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 50)        │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 50)        │          0 │ movie_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 100)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     12,928 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,237,847 (4.72 MB)

 Trainable params: 1,237,847 (4.72 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Prepare input data
train_users = train_df['user_index'].values
train_movies = train_df['movie_index'].values
train_ratings = train_df['rating'].values

# Train the neural network
model.fit([train_users, train_movies], train_ratings, epochs=10, batch_size=16, validation_split=0.2)

Epoch 1/10
63814/63814 ━━━━━━━━━━━━━━━━━━━━ 320s 5ms/step - loss: 0.7240 - mae: 0.6595 - val_loss: 0.7839 - val_mae: 0.7004
Epoch 2/10
63814/63814 ━━━━━━━━━━━━━━━━━━━━ 325s 5ms/step - loss: 0.5728 - mae: 0.5771 - val_loss: 0.8110 - val_mae: 0.7194
Epoch 3/10
63814/63814 ━━━━━━━━━━━━━━━━━━━━ 319s 5ms/step - loss: 0.5332 - mae: 0.5531 - val_loss: 0.7943 - val_mae: 0.7061
Epoch 4/10
63814/63814 ━━━━━━━━━━━━━━━━━━━━ 318s 5ms/step - loss: 0.5060 - mae: 0.5355 - val_loss: 0.8597 - val_mae: 0.7502
Epoch 5/10
63814/63814 ━━━━━━━━━━━━━━━━━━━━ 321s 5ms/step - loss: 0.4869 - mae: 0.5227 - val_loss: 0.8461 - val_mae: 0.7414
Epoch 6/10
63814/63814 ━━━━━━━━━━━━━━━━━━━━ 352s 6ms/step - loss: 0.4636 - mae: 0.5076 - val_loss: 0.9371 - val_mae: 0.7925
Epoch 7/10
63814/63814 ━━━━━━━━━━━━━━━━━━━━ 356s 6ms/step - loss: 0.4490 - mae: 0.4971 - val_loss: 0.9138 - val_mae: 0.7788
Epoch 8/10
63814/63814 ━━━━━━━━━━━━━━━━━━━━ 351s 5ms/step - loss: 0.4296 - mae: 0.4842 - val_loss: 0.8796 - val_mae: 0.7575
Epoch 9/

In [131]:
# Prepare validation data
val_users = val_df['user_index'].values
val_movies = val_df['movie_index'].values
val_ratings = val_df['rating'].values

# Evaluate model performance
loss, mae = model.evaluate([val_users, val_movies], val_ratings)
print(f"Test MAE: {mae:.4f}")

# Get predicted ratings
predicted_ratings = model.predict([val_users, val_movies])

# Compute RMSE
rmse = np.sqrt(mean_squared_error(val_ratings, predicted_ratings))
print(f"Test RMSE: {rmse:.4f}")

8541/8541 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - loss: 1.6407 - mae: 1.0255
Test MAE: 1.0240
8541/8541 ━━━━━━━━━━━━━━━━━━━━ 7s 841us/step
Test RMSE: 1.2779
